In [1]:
import html
from pathlib import Path
import pandas as pd
import ipywidgets as w
from IPython.display import display

SRC, OUT = Path("../data/gold/gold_to_label.csv"), Path("../data/gold/gold_labeled.csv")
gold = pd.read_csv(OUT if OUT.exists() else SRC)            # resumes from saved progress
gold[["intent", "priority"]] = gold[["intent", "priority"]].fillna("").astype(str)

INTENTS = ["Billing", "Technical", "Account", "Complaint", "General"]
PRIOS = ["High", "Medium", "Low"]
unlabeled = [i for i in range(len(gold)) if not gold.at[i, "intent"] or not gold.at[i, "priority"]]
state = {"i": unlabeled[0] if unlabeled else len(gold)}

guide = w.HTML("""<small>
<b>Intent: which team must act?</b> Billing = charges, refunds, payments, prices, promos ·
Technical = something not working (app, device, site, connection, outage) ·
Account = login, password, access, hacked, account details ·
Complaint = the grievance IS the point (delays, rude staff, lost items, bad experience) ·
General = questions, info, praise, chit-chat. <i>Angry about a charge → Billing.</i><br>
<b>Priority:</b> High = outage, security, money lost, time-critical, strong anger/cancel threat ·
Medium = real problem, not urgent · Low = question, feedback, praise, no time pressure.</small>""")
progress, tweet, status = w.HTML(), w.HTML(), w.HTML()
intent_btns = w.ToggleButtons(options=INTENTS, value=None, description="Intent")
prio_btns = w.ToggleButtons(options=PRIOS, value=None, description="Priority")
back_btn = w.Button(description="◀ Back")
save_btn = w.Button(description="Save & next ▶", button_style="success")

def render():
    i = state["i"]
    done = ((gold["intent"] != "") & (gold["priority"] != "")).sum()
    progress.value = f"<b>{done} / {len(gold)} labeled</b>" + (f" · row {i + 1}" if i < len(gold) else "")
    if i >= len(gold):
        tweet.value = "<h3>All rows labeled. Saved to gold_labeled.csv</h3>"
        return
    tweet.value = (f"<div style='font-size:16px;padding:12px;border:1px solid #888;border-radius:8px;"
                   f"margin:8px 0'>{html.escape(str(gold.at[i, 'text']))}</div>")
    intent_btns.value = gold.at[i, "intent"] or None
    prio_btns.value = gold.at[i, "priority"] or None

def save(_):
    i = state["i"]
    if i >= len(gold):
        return
    if not intent_btns.value or not prio_btns.value:
        status.value = "⚠️ Pick both intent and priority"
        return
    gold.at[i, "intent"], gold.at[i, "priority"] = intent_btns.value, prio_btns.value
    gold.to_csv(OUT, index=False)
    status.value = ""
    state["i"] += 1
    render()

def back(_):
    state["i"] = max(0, state["i"] - 1)
    render()

save_btn.on_click(save)
back_btn.on_click(back)
render()
display(w.VBox([guide, progress, tweet, intent_btns, prio_btns, w.HBox([back_btn, save_btn]), status]))